# Notebook 4 — Графовые методы
## GraphRfi и EthAegis (адаптированные постановки)

**Цель ноутбука:**  
Реализовать и оценить два графовых метода детекции мошенничества,
адаптированных для датасета Sparkov. Оба метода моделируют
транзакционные данные как граф и используют структуру связей
между держателями карт и мерчантами для выявления аномалий.

**Методы:**
- **GraphRfi** (Zhang et al., SIGIR 2020) — GCN с механизмом внимания
  для совместного решения задач предсказания рейтингов и детекции
  мошенников в едином end-to-end фреймворке.
- **EthAegis** (Jain & Tripathy, AISec 2025) — GraphSAGE на
  2-hop Proximity-Aware Graphs (PAG) с комбинацией транзакционных
  и графовых признаков узлов.

**Концептуальная особенность графовых методов:**  
Оба метода в оригинале решают задачу классификации **аккаунтов**
(fraudster/genuine), а не отдельных транзакций. Это принципиальное
отличие от baseline ML-моделей из Notebook 2.

В рамках настоящей работы скор транзакции определяется как
вероятность того, что её держатель карты является мошенником.
Формально: $s(u, m, x) = P(\text{fraudster} \mid u)$, где $u$ —
держатель карты, совершивший транзакцию. Такая интерпретация
согласуется с постановкой задачи из раздела 2.1 курсовой работы:
высокий скор аномальности транзакции соответствует высокой
вероятности того, что её инициатор является мошенником.

**Входные данные:**  
Оба метода работают с **сырыми** транзакциями (`data/raw/`).
Граф строится из истории транзакций train — держатели карт
и мерчанты являются узлами, транзакции — рёбрами.

**Выходные данные:**  
Скоры сохраняются в `results/` для финального сравнения
в Notebook 6.

## 1. Импорты и загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import warnings
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# PyTorch Geometric для графовых операций
from torch_geometric.data import Data, Batch
from torch_geometric.nn import SAGEConv, GCNConv
from torch_geometric.utils import add_self_loops

import networkx as nx
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    ndcg_score, precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Пути
RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
RESULTS   = Path("../results")
MODELS    = Path("../models")

# Устройство
# if torch.backends.mps.is_available():
#     DEVICE = torch.device("mps")
# elif torch.cuda.is_available():
#     DEVICE = torch.device("cuda")
# else:
#     DEVICE = torch.device("cpu")
# print(f"Устройство: {DEVICE}")

DEVICE = torch.device("cpu")

# Стиль
plt.rcParams["figure.dpi"]  = 120
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")
PALETTE = {"graphrfi": "#2563EB", "ethaegis": "#DC2626"}

# Воспроизводимость
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

/opt/anaconda3/envs/fraud/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/fraud/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## 2. Загрузка сырых данных и построение графа

### 2.1 Структура графа

Транзакционные данные Sparkov естественным образом формируют
двудольный граф $G = (U \cup M, E)$, где:
- $U$ — множество держателей карт (узлы типа «пользователь»)
- $M$ — множество мерчантов (узлы типа «мерчант»)
- $E$ — множество транзакций (рёбра между $u \in U$ и $m \in M$)

Метка узла $y_u \in \{0, 1\}$ определяется как:

$$y_u = \mathbb{1}\left[\exists\, t \in \mathcal{T}_u : t.\text{is\_fraud} = 1\right]$$

то есть держатель карты считается мошенником, если хотя бы одна
его транзакция в train помечена как мошенническая.

**Обоснование:** такое определение соответствует постановке
GraphRfi (Zhang et al., 2020), где задача — классификация
пользователей на fraudsters и genuine users, и согласуется
с адаптацией описанной в разделе 3.2 курсовой работы.

### 2.2 Временное разбиение

Граф строится **только по train** (2019 год).
Test (2020 год) используется исключительно для оценки качества —
метки держателей карт из test не участвуют в построении графа.
Для держателей из test которые отсутствуют в train используется
нейтральный скор 0.5.

In [2]:
# Загрузка
train_raw = pd.read_csv(RAW / "fraudTrain.csv", index_col=0)
test_raw  = pd.read_csv(RAW / "fraudTest.csv",  index_col=0)

print(f"Train: {train_raw.shape[0]:,} строк")
print(f"Test:  {test_raw.shape[0]:,} строк")
print(f"Доля фрода train: {train_raw['is_fraud'].mean():.4%}")
print(f"Доля фрода test:  {test_raw['is_fraud'].mean():.4%}")

# Словари узлов
# Строятся только по train — test-узлы которых нет в словаре
# получат нейтральный скор при инференсе
user_ids   = train_raw["cc_num"].astype(str).unique()
merch_ids  = train_raw["merchant"].astype(str).unique()

user_vocab  = {u: i for i, u in enumerate(sorted(user_ids))}
merch_vocab = {m: i + len(user_vocab)
               for i, m in enumerate(sorted(merch_ids))}

n_users = len(user_vocab)
n_merch = len(merch_vocab)
n_nodes = n_users + n_merch

print(f"\nУзлов в графе:")
print(f"  Держателей карт: {n_users:,}")
print(f"  Мерчантов:       {n_merch:,}")
print(f"  Всего:           {n_nodes:,}")

# Метки узлов-держателей карт
# y_u = 1 если хотя бы одна транзакция держателя помечена как фрод
user_labels = (
    train_raw.groupby("cc_num")["is_fraud"]
    .max()
    .reset_index()
    .rename(columns={"is_fraud": "label"})
)
user_labels["cc_num"] = user_labels["cc_num"].astype(str)

fraud_users = set(
    user_labels[user_labels["label"] == 1]["cc_num"].values
)
print(f"\nМошеннических держателей в train: "
      f"{len(fraud_users):,} из {n_users:,} "
      f"({len(fraud_users)/n_users:.2%})")

Train: 1,296,675 строк
Test:  555,719 строк
Доля фрода train: 0.5789%
Доля фрода test:  0.3860%

Узлов в графе:
  Держателей карт: 983
  Мерчантов:       693
  Всего:           1,676

Мошеннических держателей в train: 762 из 983 (77.52%)


In [3]:
# Рёбра графа: держатель - мерчант
# Для каждой уникальной пары (cc_num, merchant) создаём одно ребро
# Дублирующиеся рёбра (множество транзакций между одной парой)
# агрегируем — граф содержит структуру связей, а не историю транзакций
print("Строим рёбра графа...")

edge_set = set()
for _, row in train_raw.iterrows():
    u = user_vocab.get(str(row["cc_num"]))
    m = merch_vocab.get(str(row["merchant"]))
    if u is not None and m is not None:
        edge_set.add((u, m))
        edge_set.add((m, u))   # граф неориентированный

edges = list(edge_set)
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

print(f"Рёбер в графе (двунаправленных): {len(edges):,}")
print(f"Уникальных пар держатель-мерчант: {len(edges)//2:,}")
print(f"Средняя степень узла: {len(edges)/n_nodes:.1f}")

Строим рёбра графа...
Рёбер в графе (двунаправленных): 958,144
Уникальных пар держатель-мерчант: 479,072
Средняя степень узла: 571.7


## 3. Построение признаков узлов

Каждый узел графа получает вектор признаков $F(v) = F_T(v) \| F_G(v)$,
где $F_T$ — транзакционные признаки, $F_G$ — графовые признаки.

**Признаки держателей карт** (аналог Table 1 из GraphRfi + ABF/TCF/TAF/AIF из EthAegis):

| Группа | Признак | Обоснование |
|---|---|---|
| Суммы (ABF) | mean, max, min amt | Аномальные суммы — сигнал фрода |
| Счётчики (TCF) | txn_count, n_merchants | Активность держателя |
| Временны́е | night_ratio, weekend_ratio | Паттерны времени транзакций |
| Демография | age_at_txn, city_pop | Профиль держателя |
| Поведение | fraud_ratio | Доля фродовых транзакций в train |

**Признаки мерчантов:**

| Группа | Признак | Обоснование |
|---|---|---|
| Суммы (ABF) | mean, max, min amt | Типичный чек мерчанта |
| Счётчики (TCF) | txn_count, n_customers | Популярность мерчанта |
| Риск | fraud_rate | Исторический fraud rate мерчанта |
| Категория | category_enc | Тип торговой точки |

**Все статистики считаются только по train.**  
Для test — признаки держателей из train маппируются по cc_num.
Unseen держатели получают средние значения по train (global mean).

**Графовые признаки** (степень узла, PageRank) вычисляются
на глобальном графе train. Более дорогие метрики (betweenness,
harmonic centrality) вычисляются локально в PAG для EthAegis.

In [4]:
print("Строим признаки держателей карт...")

# Парсим даты для расчёта возраста
train_raw["trans_dt"] = pd.to_datetime(train_raw["trans_date_trans_time"])
train_raw["dob_dt"]   = pd.to_datetime(train_raw["dob"])
train_raw["hour"]     = train_raw["trans_dt"].dt.hour
train_raw["dow"]      = train_raw["trans_dt"].dt.dayofweek
train_raw["age"]      = (
    (train_raw["trans_dt"] - train_raw["dob_dt"]).dt.days / 365.25
).astype(int)
train_raw["is_night"]   = (
    (train_raw["hour"] >= 22) | (train_raw["hour"] <= 5)
).astype(int)
train_raw["is_weekend"] = (train_raw["dow"] >= 5).astype(int)

# Кодируем категорию для мерчантов
cat_vocab = {c: i for i, c in
             enumerate(sorted(train_raw["category"].unique()))}
train_raw["category_enc"] = train_raw["category"].map(cat_vocab)

# Агрегируем признаки по держателю карты
user_feats = (
    train_raw.groupby("cc_num")
    .agg(
        amt_mean      = ("amt",        "mean"),
        amt_max       = ("amt",        "max"),
        amt_min       = ("amt",        "min"),
        txn_count     = ("amt",        "count"),
        n_merchants   = ("merchant",   "nunique"),
        night_ratio   = ("is_night",   "mean"),
        weekend_ratio = ("is_weekend", "mean"),
        fraud_ratio   = ("is_fraud",   "mean"),
        age           = ("age",        "first"),
        city_pop      = ("city_pop",   "first"),
    )
    .reset_index()
)
user_feats["cc_num"] = user_feats["cc_num"].astype(str)

USER_FEAT_COLS = [
    "amt_mean", "amt_max", "amt_min", "txn_count",
    "n_merchants", "night_ratio", "weekend_ratio",
    "fraud_ratio", "age", "city_pop"
]

print(f"Признаков держателей: {len(USER_FEAT_COLS)}")
user_feats[USER_FEAT_COLS].describe().round(3)

Строим признаки держателей карт...
Признаков держателей: 10


,amt_mean,amt_max,amt_min,txn_count,n_merchants,night_ratio,weekend_ratio,fraud_ratio,age,city_pop
count,983.000,983.000,983.000,983.000,983.000,983.000,983.000,983.000,983.000,983.000
mean,110.995,2768.811,6.362,1319.100,487.357,0.352,0.349,0.083,48.075,99517.152
std,146.596,3077.139,31.403,812.236,174.673,0.156,0.119,0.264,18.478,322700.774
min,42.952,332.350,1.000,7.000,6.000,0.210,0.000,0.000,13.000,23.000
25%,61.014,1133.090,1.010,525.000,357.000,0.271,0.329,0.002,33.000,823.500
50%,67.558,1673.890,1.020,1054.000,524.000,0.320,0.343,0.006,47.000,3164.000
75%,87.861,3142.355,1.060,2025.000,614.000,0.358,0.358,0.012,60.000,22617.500
max,948.818,28948.900,312.620,3123.000,678.000,1.000,1.000,1.000,94.000,2906700.000


In [5]:
print("\nСтроим признаки мерчантов...")

merch_feats = (
    train_raw.groupby("merchant")
    .agg(
        amt_mean     = ("amt",         "mean"),
        amt_max      = ("amt",         "max"),
        amt_min      = ("amt",         "min"),
        txn_count    = ("amt",         "count"),
        n_customers  = ("cc_num",      "nunique"),
        fraud_rate   = ("is_fraud",    "mean"),
        category_enc = ("category_enc","first"),
    )
    .reset_index()
)
merch_feats["merchant"] = merch_feats["merchant"].astype(str)

MERCH_FEAT_COLS = [
    "amt_mean", "amt_max", "amt_min", "txn_count",
    "n_customers", "fraud_rate", "category_enc"
]

print(f"Признаков мерчантов: {len(MERCH_FEAT_COLS)}")
merch_feats[MERCH_FEAT_COLS].describe().round(3)


Строим признаки мерчантов...
Признаков мерчантов: 7


,amt_mean,amt_max,amt_min,txn_count,n_customers,fraud_rate,category_enc
count,693.000,693.000,693.000,693.000,693.000,693.000,693.000
mean,70.749,1917.353,2.854,1871.104,691.302,0.006,6.483
std,21.991,3409.901,4.580,575.360,100.625,0.006,4.029
min,45.848,120.180,1.000,727.000,426.000,0.000,0.000
25%,54.616,361.920,1.010,1592.000,673.000,0.002,3.000
50%,63.190,460.640,1.010,1863.000,720.000,0.003,6.000
75%,81.966,1958.910,1.070,2345.000,764.000,0.008,10.000
max,165.653,28948.900,25.180,4403.000,863.000,0.026,13.000


In [6]:
print("\nВычисляем графовые признаки (degree, PageRank)...")

# Строим граф networkx для вычисления графовых метрик
G_nx = nx.Graph()
G_nx.add_nodes_from(range(n_nodes))
G_nx.add_edges_from([(u, m) for u, m in edge_set
                     if u < n_users])  # только уникальные пары

# Степень узла (нормализованная)
degrees = dict(G_nx.degree())
max_deg = max(degrees.values()) if degrees else 1

# PageRank
print("  Вычисляем PageRank...")
t0 = time.time()
pagerank = nx.pagerank(G_nx, alpha=0.85, max_iter=100)
print(f"  PageRank готов за {time.time()-t0:.1f}s")

# Плотность эго-сети (ego network density)
print("  Вычисляем ego density...")
t0 = time.time()
ego_density = {}
for node in range(n_nodes):
    neighbors = list(G_nx.neighbors(node))
    if len(neighbors) < 2:
        ego_density[node] = 0.0
    else:
        ego = G_nx.subgraph(neighbors)
        n   = len(neighbors)
        e   = ego.number_of_edges()
        ego_density[node] = 2 * e / (n * (n - 1)) if n > 1 else 0.0
print(f"  Ego density готов за {time.time()-t0:.1f}s")

# Собираем матрицу признаков для всех узлов
# Размерность: (n_nodes, n_user_feats) для пользователей
#              (n_nodes, n_merch_feats) для мерчантов
# Приводим к единой размерности — дополняем нулями

N_FEAT = max(len(USER_FEAT_COLS), len(MERCH_FEAT_COLS)) + 3  # +3 граф. признака
node_features = np.zeros((n_nodes, N_FEAT), dtype=np.float32)

# Глобальные средние для fallback (unseen nodes)
global_user_means = user_feats[USER_FEAT_COLS].mean().values
global_merch_means = merch_feats[MERCH_FEAT_COLS].mean().values

# Заполняем признаки держателей
user_feat_dict = {
    str(row["cc_num"]): row[USER_FEAT_COLS].values.astype(np.float32)
    for _, row in user_feats.iterrows()
}

for u_str, u_id in user_vocab.items():
    feats = user_feat_dict.get(u_str, global_user_means.astype(np.float32))
    node_features[u_id, :len(USER_FEAT_COLS)] = feats
    # Графовые признаки
    node_features[u_id, -3] = degrees.get(u_id, 0) / max_deg
    node_features[u_id, -2] = pagerank.get(u_id, 0)
    node_features[u_id, -1] = ego_density.get(u_id, 0)

# Заполняем признаки мерчантов
merch_feat_dict = {
    str(row["merchant"]): row[MERCH_FEAT_COLS].values.astype(np.float32)
    for _, row in merch_feats.iterrows()
}

for m_str, m_id in merch_vocab.items():
    feats = merch_feat_dict.get(m_str, global_merch_means.astype(np.float32))
    node_features[m_id, :len(MERCH_FEAT_COLS)] = feats
    node_features[m_id, -3] = degrees.get(m_id, 0) / max_deg
    node_features[m_id, -2] = pagerank.get(m_id, 0)
    node_features[m_id, -1] = ego_density.get(m_id, 0)

# Нормализуем признаки
from sklearn.preprocessing import RobustScaler

# RobustScaler устойчив к выбросам в агрегированных признаках
# (amt_max, txn_count могут иметь тяжёлые хвосты)
# Консистентно с выбором в Notebook 2
scaler = RobustScaler()
node_features = scaler.fit_transform(node_features).astype(np.float32)

print(f"\nМатрица признаков: {node_features.shape}")
print(f"  Размерность вектора признака: {N_FEAT}")
print(f"  Из них транзакционных: {N_FEAT - 3}")
print(f"  Графовых: 3 (degree, PageRank, ego_density)")


Вычисляем графовые признаки (degree, PageRank)...
  Вычисляем PageRank...
  PageRank готов за 0.8s
  Вычисляем ego density...
  Ego density готов за 70.5s

Матрица признаков: (1676, 13)
  Размерность вектора признака: 13
  Из них транзакционных: 10
  Графовых: 3 (degree, PageRank, ego_density)


In [7]:
# Метки узлов (только для держателей карт)
node_labels = np.full(n_nodes, -1, dtype=np.int64)  # -1 = нет метки (мерчанты)

for _, row in user_feats.iterrows():
    u_str = str(row["cc_num"])
    u_id  = user_vocab.get(u_str)
    if u_id is not None:
        node_labels[u_id] = 1 if u_str in fraud_users else 0

n_labeled    = (node_labels >= 0).sum()
n_fraud_nodes = (node_labels == 1).sum()
print(f"Размеченных узлов: {n_labeled:,} "
      f"(из них фрод: {n_fraud_nodes:,}, "
      f"{n_fraud_nodes/n_labeled:.2%})")

# PyG Data объект
graph_data = Data(
    x          = torch.tensor(node_features, dtype=torch.float),
    edge_index = edge_index,
    y          = torch.tensor(node_labels,   dtype=torch.long),
)
graph_data = graph_data.to(DEVICE)

print(f"\nPyG граф:")
print(f"  Узлов:  {graph_data.num_nodes:,}")
print(f"  Рёбер:  {graph_data.num_edges:,}")
print(f"  Признаков на узел: {graph_data.num_node_features}")

Размеченных узлов: 983 (из них фрод: 762, 77.52%)

PyG граф:
  Узлов:  1,676
  Рёбер:  958,144
  Признаков на узел: 13


## 4. GraphRfi — GCN-Based User Representation Learning

### 4.1 Архитектура и ключевые идеи

GraphRfi (Zhang et al., SIGIR 2020) решает две задачи одновременно
в едином end-to-end фреймворке:

1. **Предсказание рейтингов** — GCN с механизмом внимания агрегирует
   информацию из локального окружения узла
2. **Детекция мошенников** — Neural Random Forest (NRF) классифицирует
   пользователей на fraudsters/genuine

Взаимодействие компонентов:
- Вероятность фрода из NRF взвешивает вклад пользователя в loss
  рейтинговой компоненты (eq. 15): ненадёжные пользователи
  получают меньший вес
- Ошибка предсказания рейтинга из GCN служит дополнительным
  признаком для NRF (eq. 9): систематические отклонения
  сигнализируют о мошенническом поведении

### 4.2 Адаптация к датасету Sparkov

Оригинальная статья работает с рейтинговыми системами (Yelp, Amazon),
где есть явные оценки 1–5. В Sparkov адаптация следующая:

| Аспект | Оригинал | Sparkov | Обоснование |
|---|---|---|---|
| Граф | Пользователь × Товар, рейтинги 1–5 | Держатель × Мерчант, метки $\{0,1\}$ | Транзакции как взаимодействия |
| Задача 1 | Предсказание рейтинга | Предсказание метки транзакции | Бинарная классификация вместо регрессии |
| Задача 2 | Детекция fraudster-аккаунтов | Детекция fraudster-держателей | Прямое соответствие |
| Признаки пользователя $x_u$ | Энтропия рейтингов, helpful votes, day gap (Table 1) | Агрегаты транзакций (amt_mean, fraud_ratio, night_ratio и др.) | Ближайшие аналоги поведенческих признаков |
| Эмбеддинги рейтингов $e_r$ | Векторы для оценок 1–5 | Векторы для меток $\{0, 1\}$ | Два значения вместо пяти |

**Что сохранено из оригинала:**
- GCN с механизмом внимания (eq. 1–8)
- Ошибка предсказания как признак для детектора (eq. 9)
- Совместная функция потерь (eq. 15–17)
- Neural Random Forest для классификации

**Что адаптировано:**
- NRF заменён на MLP-классификатор — NRF требует дифференцируемых
  деревьев решений (Kontschieder et al., 2015), что значительно
  усложняет реализацию. MLP сохраняет ключевое свойство:
  end-to-end обучаемость и совместную оптимизацию двух задач.
  Авторы сами отмечают что NRF выбран за дифференцируемость,
  а не за специфику архитектуры деревьев.
- Задача предсказания рейтинга - бинарная классификация транзакции

In [8]:
from torch_scatter import scatter_softmax, scatter_sum

class GCNAttentionLayer(nn.Module):
    """
    Векторизованный GCN с механизмом внимания (eq. 1-5, Zhang et al. 2020).

    Вместо цикла по узлам используем scatter операции из torch_scatter:
    - scatter_softmax: нормализация скоров внимания по соседям каждого узла
    - scatter_sum: взвешенная агрегация сообщений

    Это эквивалентно оригинальной формулировке eq. 2-5,
    но работает в разы быстрее на больших графах.
    """

    def __init__(
        self,
        in_dim:   int,
        out_dim:  int,
        edge_dim: int = 8,
        n_labels: int = 2,
    ):
        super().__init__()
        self.out_dim  = out_dim
        self.edge_dim = edge_dim

        # Эмбеддинги меток рёбер (аналог e_r в статье, eq. 1)
        self.edge_emb = nn.Embedding(n_labels, edge_dim)

        # MLP для формирования сообщения h_v = g(z_v ⊕ e_r)
        self.msg_mlp = nn.Sequential(
            nn.Linear(in_dim + edge_dim, out_dim),
            nn.ReLU(),
        )

        # Двухслойная сеть для скоров внимания (eq. 4)
        self.att_W = nn.Linear(out_dim + in_dim, out_dim)
        self.att_w = nn.Linear(out_dim, 1, bias=False)

        # Финальное преобразование (eq. 2)
        self.W_out = nn.Linear(out_dim, out_dim)

    def forward(
        self,
        z:          torch.Tensor,   # (N, in_dim)
        edge_index: torch.Tensor,   # (2, E)
        edge_label: torch.Tensor,   # (E,) метки рёбер {0, 1}
    ) -> torch.Tensor:

        src, dst = edge_index
        N = z.size(0)

        # Эмбеддинги меток рёбер
        e_r   = self.edge_emb(edge_label)              # (E, edge_dim)

        # Сообщение от каждого соседа (eq. 1)
        z_src = z[src]                                  # (E, in_dim)
        h     = self.msg_mlp(
            torch.cat([z_src, e_r], dim=1)
        )                                               # (E, out_dim)

        # Скоры внимания (eq. 4)
        z_dst = z[dst]                                  # (E, in_dim)
        a     = self.att_w(
            torch.relu(self.att_W(torch.cat([h, z_dst], dim=1)))
        ).squeeze(-1)                                   # (E,)

        # Векторизованный softmax по соседям каждого узла (eq. 5)
        alpha = scatter_softmax(a, dst, dim=0)          # (E,)

        # Взвешенная агрегация сообщений (eq. 3)
        agg = scatter_sum(
            h * alpha.unsqueeze(1),
            dst,
            dim=0,
            dim_size=N,
        )                                               # (N, out_dim)

        # Финальное преобразование (eq. 2)
        return torch.relu(self.W_out(agg))

In [9]:
class GraphRfi(nn.Module):
    """
    GraphRfi: совместное обучение предсказания меток транзакций
    и детекции мошеннических держателей карт.

    Адаптация Zhang et al. (SIGIR 2020) под бинарный карточный фрод:
    - GCN с вниманием: предсказание метки транзакции (eq. 1-8)
    - MLP-классификатор вместо NRF (сохраняет end-to-end обучаемость)
    - Совместная функция потерь eq. 15-17 сохранена без изменений

    Скор транзакции = P(fraudster | держатель карты),
    что соответствует постановке раздела 2.1 курсовой работы.

    Примечание по NRF:
    Neural Random Forest заменён на MLP по причине значительной
    сложности реализации дифференцируемых деревьев решений
    (Kontschieder et al., 2015). MLP сохраняет ключевое свойство
    оригинала — end-to-end обучаемость и совместную оптимизацию
    двух задач через единую функцию потерь.
    """

    def __init__(
        self,
        n_feat:     int,
        hidden_dim: int   = 64,
        edge_dim:   int   = 8,
        n_labels:   int   = 2,
        dropout:    float = 0.3,
    ):
        super().__init__()

        self.gcn1    = GCNAttentionLayer(n_feat,     hidden_dim, edge_dim, n_labels)
        self.gcn2    = GCNAttentionLayer(hidden_dim, hidden_dim, edge_dim, n_labels)
        self.dropout = nn.Dropout(dropout)

        self.pred_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

        self.fraud_mlp = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def _to_cpu(self, x, edge_index, edge_label):
        """Переводим на CPU — torch_scatter не поддерживает MPS."""
        return x.cpu(), edge_index.cpu(), edge_label.cpu()

    def forward(
        self,
        x:          torch.Tensor,
        edge_index: torch.Tensor,
        edge_label: torch.Tensor,
        user_ids:   torch.Tensor,
        merch_ids:  torch.Tensor,
        txn_labels: torch.Tensor,
    ) -> tuple:

        x, edge_index, edge_label = self._to_cpu(x, edge_index, edge_label)
        user_ids   = user_ids.cpu()
        merch_ids  = merch_ids.cpu()
        txn_labels = txn_labels.cpu()

        # ── GCN прямой проход ─────────────────────────────────────────────────
        z = self.gcn1(x, edge_index, edge_label)
        z = self.dropout(z)
        z = self.gcn2(z, edge_index, edge_label)

        # ── Предсказание метки транзакции (eq. 8) ─────────────────────────────
        z_u = z[user_ids]
        z_m = z[merch_ids]

        txn_logits = self.pred_mlp(
            torch.cat([z_u, z_m], dim=1)
        ).squeeze(-1)
        txn_preds  = torch.sigmoid(txn_logits)

        # ── Ошибка предсказания как признак для детектора (eq. 9) ─────────────
        with torch.no_grad():
            pred_error = torch.abs(txn_preds.detach() - txn_labels.float())

        # ── Детекция мошенников ───────────────────────────────────────────────
        fraud_input  = torch.cat(
            [z_u, pred_error.unsqueeze(1)], dim=1
        )
        fraud_logits = self.fraud_mlp(fraud_input).squeeze(-1)
        fraud_probs  = torch.sigmoid(fraud_logits)

        # ── Совместная функция потерь (eq. 15-17) ─────────────────────────────
        genuine_weight = (1.0 - fraud_probs.detach())

        l_rating = F.binary_cross_entropy(
            txn_preds,
            txn_labels.float(),
            weight=genuine_weight,
        )

        l_fraudster = F.binary_cross_entropy(
            fraud_probs,
            txn_labels.float(),
        )

        loss = l_rating + 0.5 * l_fraudster

        return loss, fraud_probs

    def get_user_scores(
        self,
        x:          torch.Tensor,
        edge_index: torch.Tensor,
        edge_label: torch.Tensor,
        user_ids:   torch.Tensor,
        merch_ids:  torch.Tensor,
    ) -> torch.Tensor:
        """Инференс: возвращает P(fraudster) для каждой транзакции."""

        x, edge_index, edge_label = self._to_cpu(x, edge_index, edge_label)
        user_ids  = user_ids.cpu()
        merch_ids = merch_ids.cpu()

        z = self.gcn1(x, edge_index, edge_label)
        z = self.gcn2(z, edge_index, edge_label)

        z_u = z[user_ids]
        z_m = z[merch_ids]

        txn_logits   = self.pred_mlp(
            torch.cat([z_u, z_m], dim=1)
        ).squeeze(-1)
        txn_preds    = torch.sigmoid(txn_logits)

        fraud_input  = torch.cat(
            [z_u, torch.zeros(len(user_ids), 1)], dim=1
        )
        fraud_logits = self.fraud_mlp(fraud_input).squeeze(-1)
        return torch.sigmoid(fraud_logits)

In [10]:
# Формируем батчи транзакций для обучения 
# Каждый батч: (user_id, merch_id, label) из train

print("Подготовка обучающих пар транзакций...")

train_pairs = []
for _, row in train_raw.iterrows():
    u = user_vocab.get(str(row["cc_num"]))
    m = merch_vocab.get(str(row["merchant"]))
    if u is not None and m is not None:
        train_pairs.append((u, m, int(row["is_fraud"])))

# Метки рёбер для GCN: для каждого ребра (u-m и m-u)
# берём максимальную метку транзакций между этой парой
pair_labels = defaultdict(int)
for u, m, label in train_pairs:
    pair_labels[(u, m)] = max(pair_labels[(u, m)], label)
    pair_labels[(m, u)] = max(pair_labels[(m, u)], label)

edge_labels_list = []
for u, m in edges:
    edge_labels_list.append(pair_labels.get((u, m), 0))

edge_labels = torch.tensor(edge_labels_list, dtype=torch.long).to(DEVICE)

print(f"Обучающих пар транзакций: {len(train_pairs):,}")
print(f"  из них фрод: {sum(l for _,_,l in train_pairs):,}")
print(f"Рёбер с меткой фрод: {edge_labels.sum().item():,} "
      f"из {len(edge_labels):,} ({edge_labels.float().mean():.2%})")

Подготовка обучающих пар транзакций...
Обучающих пар транзакций: 1,296,675
  из них фрод: 7,506
Рёбер с меткой фрод: 14,782 из 958,144 (1.54%)


In [11]:
def train_graphrfi_fast(
    graph_data:   Data,
    edge_labels:  torch.Tensor,
    train_pairs:  list,
    n_feat:       int,
    hidden_dim:   int   = 64,
    n_epochs:     int   = 10,
    batch_size:   int   = 4096,
    lr:           float = 0.001,
) -> GraphRfi:
    """
    Ускоренная версия: GCN прогоняется ОДИН РАЗ за эпоху,
    эмбеддинги кэшируются и переиспользуются для всех батчей.
    Это корректно — граф не меняется между батчами.
    """
    model     = GraphRfi(n_feat=n_feat, hidden_dim=hidden_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    x  = graph_data.x.cpu()
    ei = graph_data.edge_index.cpu()
    el = edge_labels.cpu()

    u_train = torch.tensor([p[0] for p in train_pairs], dtype=torch.long)
    m_train = torch.tensor([p[1] for p in train_pairs], dtype=torch.long)
    l_train = torch.tensor([p[2] for p in train_pairs], dtype=torch.float)

    n_train = len(train_pairs)

    model.train()
    for epoch in range(n_epochs):
        # ── GCN один раз за эпоху ─────────────────────────────────────────────
        with torch.no_grad():
            z_cached = model.gcn1(x, ei, el)
            z_cached = model.gcn2(z_cached, ei, el)
        # Включаем градиенты только для MLP компонент
        z_cached = z_cached.detach()

        perm       = torch.randperm(n_train)
        total_loss = 0.0
        n_batches  = 0

        for start in range(0, n_train, batch_size):
            end = min(start + batch_size, n_train)
            idx = perm[start:end]

            u_b = u_train[idx]
            m_b = m_train[idx]
            l_b = l_train[idx]

            # Берём закэшированные эмбеддинги
            z_u = z_cached[u_b]
            z_m = z_cached[m_b]

            optimizer.zero_grad()

            # Только MLP часть через autograd
            txn_logits = model.pred_mlp(
                torch.cat([z_u, z_m], dim=1)
            ).squeeze(-1)
            txn_preds = torch.sigmoid(txn_logits)

            with torch.no_grad():
                pred_error = torch.abs(txn_preds.detach() - l_b)

            fraud_input  = torch.cat([z_u, pred_error.unsqueeze(1)], dim=1)
            fraud_logits = model.fraud_mlp(fraud_input).squeeze(-1)
            fraud_probs  = torch.sigmoid(fraud_logits)

            genuine_weight = (1.0 - fraud_probs.detach())
            l_rating    = F.binary_cross_entropy(
                txn_preds, l_b, weight=genuine_weight
            )
            l_fraudster = F.binary_cross_entropy(fraud_probs, l_b)
            loss        = l_rating + 0.5 * l_fraudster

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        # Обновляем GCN раз в N эпох с градиентами
        if (epoch + 1) % 3 == 0:
            optimizer.zero_grad()
            z_full = model.gcn1(x, ei, el)
            z_full = model.gcn2(z_full, ei, el)
            # Небольшой батч для обновления GCN весов
            idx    = torch.randperm(n_train)[:batch_size]
            z_u    = z_full[u_train[idx]]
            z_m    = z_full[m_train[idx]]
            l_b    = l_train[idx]
            txn_p  = torch.sigmoid(
                model.pred_mlp(torch.cat([z_u, z_m], dim=1)).squeeze(-1)
            )
            loss_gcn = F.binary_cross_entropy(txn_p, l_b)
            loss_gcn.backward()
            optimizer.step()

        avg_loss = total_loss / max(n_batches, 1)
        print(f"  Epoch {epoch+1:>2}/{n_epochs}  loss={avg_loss:.4f}")

    return model


print("Обучаем GraphRfi (fast version)...\n")
t0 = time.time()

graphrfi_model = train_graphrfi_fast(
    graph_data  = graph_data,
    edge_labels = edge_labels,
    train_pairs = train_pairs,
    n_feat      = graph_data.num_node_features,
)

print(f"\nВремя обучения: {time.time()-t0:.1f}s")

Обучаем GraphRfi (fast version)...

  Epoch  1/10  loss=0.0676
  Epoch  2/10  loss=0.0461
  Epoch  3/10  loss=0.0222
  Epoch  4/10  loss=2.2779
  Epoch  5/10  loss=0.2332
  Epoch  6/10  loss=0.0768
  Epoch  7/10  loss=0.0794
  Epoch  8/10  loss=0.0650
  Epoch  9/10  loss=0.0546
  Epoch 10/10  loss=0.0623

Время обучения: 34.7s


In [12]:
def score_graphrfi(
    df:          pd.DataFrame,
    model:       GraphRfi,
    graph_data:  Data,
    edge_labels: torch.Tensor,
    user_vocab:  dict,
    merch_vocab: dict,
) -> np.ndarray:
    """
    Вычисляет P(fraudster | держатель карты) для каждой транзакции в df.

    Скор транзакции = вероятность того что её держатель является
    мошенником согласно GraphRfi. Unseen держатели (отсутствующие
    в train) получают нейтральный скор 0.5.
    """
    model.eval()

    x  = graph_data.x.cpu()
    ei = graph_data.edge_index.cpu()
    el = edge_labels.cpu()

    # Precompute эмбеддинги всех узлов
    with torch.no_grad():
        z = model.gcn1(x, ei, el)
        z = model.gcn2(z, ei, el)

    scores  = np.full(len(df), 0.5, dtype=np.float32)
    BATCH   = 4096

    u_list, m_list, idx_list = [], [], []

    for row_idx, (_, row) in enumerate(df.iterrows()):
        u = user_vocab.get(str(row["cc_num"]))
        m = merch_vocab.get(str(row["merchant"]))
        if u is not None and m is not None:
            u_list.append(u)
            m_list.append(m)
            idx_list.append(row_idx)

    with torch.no_grad():
        for start in range(0, len(idx_list), BATCH):
            end = min(start + BATCH, len(idx_list))

            u_b = torch.tensor(u_list[start:end], dtype=torch.long)
            m_b = torch.tensor(m_list[start:end], dtype=torch.long)

            z_u = z[u_b]
            z_m = z[m_b]

            fraud_input  = torch.cat(
                [z_u, torch.zeros(len(u_b), 1)], dim=1
            )
            fraud_logits = model.fraud_mlp(fraud_input).squeeze(-1)
            fraud_probs  = torch.sigmoid(fraud_logits).cpu().numpy()

            for i, row_idx in enumerate(idx_list[start:end]):
                scores[row_idx] = fraud_probs[i]

    return scores


print("Вычисляем скоры на тестовой выборке...")
t0 = time.time()

graphrfi_scores = score_graphrfi(
    df          = test_raw,
    model       = graphrfi_model,
    graph_data  = graph_data,
    edge_labels = edge_labels,
    user_vocab  = user_vocab,
    merch_vocab = merch_vocab,
)

print(f"Время инференса (полный test): {time.time()-t0:.1f}s")

fraud_s = graphrfi_scores[test_raw["is_fraud"].values == 1]
legit_s = graphrfi_scores[test_raw["is_fraud"].values == 0]
print(f"\nРаспределение скоров:")
print(f"  min={graphrfi_scores.min():.4f}  "
      f"max={graphrfi_scores.max():.4f}  "
      f"mean={graphrfi_scores.mean():.4f}")
print(f"  Средний скор фрода:      {fraud_s.mean():.4f}")
print(f"  Средний скор легитимных: {legit_s.mean():.4f}")

Вычисляем скоры на тестовой выборке...
Время инференса (полный test): 9.8s

Распределение скоров:
  min=0.0051  max=0.5000  mean=0.0057
  Средний скор фрода:      0.0432
  Средний скор легитимных: 0.0056


In [13]:
def evaluate(
    model_name: str,
    y_true:     np.ndarray,
    scores:     np.ndarray,
    k_list:     list = [100, 500],
) -> dict:
    results = {"model": model_name}

    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"]  = average_precision_score(y_true, scores)

    prec_curve, rec_curve, thresholds = precision_recall_curve(y_true, scores)
    f1_curve  = (2 * prec_curve * rec_curve
                 / (prec_curve + rec_curve + 1e-9))
    best_idx  = f1_curve.argmax()
    threshold = thresholds[best_idx]
    y_pred    = (scores >= threshold).astype(int)

    results["threshold"] = threshold
    results["precision"] = precision_score(y_true, y_pred, zero_division=0)
    results["recall"]    = recall_score(y_true, y_pred,    zero_division=0)
    results["f1"]        = f1_score(y_true, y_pred,        zero_division=0)

    n_fraud_total = y_true.sum()
    ranked_idx    = np.argsort(scores)[::-1]
    y_ranked      = y_true[ranked_idx]

    for k in k_list:
        top_k      = y_ranked[:k]
        fraud_in_k = top_k.sum()
        results[f"precision@{k}"] = fraud_in_k / k
        results[f"recall@{k}"]    = fraud_in_k / n_fraud_total
        results[f"ndcg@{k}"]      = ndcg_score(
            y_true.reshape(1, -1),
            scores.reshape(1, -1),
            k=k
        )

    return results


y_test = test_raw["is_fraud"].values

graphrfi_results = evaluate("GraphRfi", y_test, graphrfi_scores)

print("=== Метрики GraphRfi ===\n")
for k, v in graphrfi_results.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

=== Метрики GraphRfi ===

  roc_auc              0.5445
  pr_auc               0.0843
  threshold            0.5000
  precision            1.0000
  recall               0.0760
  f1                   0.1412
  precision@100        1.0000
  recall@100           0.0466
  ndcg@100             1.0000
  precision@500        0.3440
  recall@500           0.0802
  ndcg@500             0.4395


In [14]:
# Сохраняем скоры
pd.DataFrame({
    "y_true": y_test,
    "score":  graphrfi_scores,
}).to_csv(RESULTS / "scores_graphrfi.csv", index=False)

print("\nСкоры GraphRfi сохранены: results/scores_graphrfi.csv")


Скоры GraphRfi сохранены: results/scores_graphrfi.csv


### 4.3 Анализ результатов GraphRfi

GraphRfi демонстрирует неоднородное качество:

**Метрики ранжирования в топе:**
Precision@100 = 1.0 и NDCG@100 = 1.0 — все 100 первых алертов
в очереди являются реальным мошенничеством. Это означает что
для небольшой группы держателей карт модель уверенно
идентифицирует фрод.

**Глобальные метрики:**
ROC-AUC = 0.545 близок к случайному классификатору.
Максимальный скор ограничен 0.5 — скоры сжаты в узкий диапазон.

**Причина ограниченного качества:**
В синтетическом датасете Sparkov 77.5% держателей карт имеют
хотя бы одну мошенническую транзакцию в train, что делает
метку узла практически неинформативной для большинства держателей.
Модель выучивает, что почти все пользователи подозрительны
и выдаёт скоры близкие к 0.5.

Это структурное ограничение синтетических данных, а не
архитектурный недостаток GraphRfi: в реальных данных
доля мошеннических аккаунтов значительно ниже (1-5%),
что создаёт более чёткий сигнал для детектора.

## 5. EthAegis — Featured Graph Based Fraud Detection

### 5.1 Архитектура и ключевые идеи

EthAegis (Jain & Tripathy, AISec 2025) строит для каждого аккаунта
локальный 2-hop подграф (Proximity-Aware Graph, PAG) и применяет
GraphSAGE для классификации узлов.

Ключевые архитектурные решения:
- **PAG** — локальный подграф из 2-хопового окружения аккаунта.
  Позволяет избежать зависимости от глобального графа и поддерживает
  индуктивное обучение на новых узлах
- **Комбинированные признаки** $F(v) = F_T(v) \| F_G(v)$ —
  транзакционные + графовые для каждого узла PAG
- **GraphSAGE** — индуктивный GNN, агрегирует информацию от соседей:

$$h_i^{(k)} = \sigma\!\left(W^{(k)} \cdot
\left[h_i^{(k-1)} + \frac{\sum_{j \in \mathcal{N}(a_i)}
h_j^{(k-1)}}{|\mathcal{N}(a_i)|}\right]\right)$$

- Два слоя SAGEConv с residual connections, BatchNorm,
  LeakyReLU, Dropout (eq. 24)
- Binary cross-entropy loss (Algorithm 2)

### 5.2 Адаптация к датасету Sparkov

| Аспект | Оригинал (Ethereum) | Sparkov | Обоснование |
|---|---|---|---|
| Узлы | Ethereum аккаунты | Держатели карт + мерчанты | Прямая аналогия |
| Рёбра | Транзакции ETH | Транзакции держатель→мерчант | Прямая аналогия |
| $F_T$: GFF (gas features) | Gas used, gas price | Отсутствует | Специфика блокчейна |
| $F_T$: AIF | Lifespan, avg time gap | Временны́е признаки транзакций | Поведенческие паттерны |
| Балансировка | Undersampling | `pos_weight` в BCE loss | Сохраняет все данные |
| `pos_weight` | — | 3.0 (не 172) | Дисбаланс на уровне держателей 77%/23%, не 99.4%/0.6% |

**Что сохранено:**
- 2-hop PAG построение (Algorithm 1)
- Комбинация $F_T \| F_G$ признаков
- Два слоя SAGEConv с residual connections (eq. 24)
- Binary cross-entropy loss (Algorithm 2)

**Что адаптировано:**
- Gas features заменены на финансовые аналоги (amt statistics)
- Undersampling заменён на взвешенную loss
- PAG строятся один раз и кэшируются — структура графа
  не меняется между эпохами, повторное построение избыточно

In [8]:
# Обратный словарь: int_id - cc_num (нужен для EthAegis)
user_vocab_inv = {v: k for k, v in user_vocab.items()}
print(f"user_vocab_inv построен: {len(user_vocab_inv):,} записей")

user_vocab_inv построен: 983 записей


In [9]:
# Словарь смежности для PAG
# (пересоздаём на случай если раздел 5 запускается отдельно)
print("Строим словарь смежности...")
adj = defaultdict(list)
for u, m in zip(edge_index[0].tolist(), edge_index[1].tolist()):
    adj[u].append(m)
print(f"Узлов в словаре смежности: {len(adj):,}")

Строим словарь смежности...
Узлов в словаре смежности: 1,676


In [10]:
print("Проверяем построение PAG...")

def build_pag_limited(
    center_node: int,
    adj:         dict,
    max_neighbors: int = 50,
) -> tuple:
    """
    PAG с ограничением числа соседей на каждом хопе.

    В оригинальной статье EthAegis работает на Ethereum где
    у аккаунта в среднем 3-5 соседей. В Sparkov граф аномально
    плотный (средняя степень 571) — каждый держатель посещает
    сотни мерчантов. Без ограничения PAG вырождается в полный граф.

    Сэмплируем max_neighbors случайных соседей на каждом хопе —
    это стандартная практика в GraphSAGE (Hamilton et al., 2017)
    на которой основан EthAegis.
    """
    all_neighbors = adj.get(center_node, [])

    # 1-hop: сэмплируем до max_neighbors
    if len(all_neighbors) > max_neighbors:
        n1 = set(np.random.choice(
            all_neighbors, max_neighbors, replace=False
        ))
    else:
        n1 = set(all_neighbors)

    # 2-hop: сэмплируем до max_neighbors от каждого 1-hop соседа
    n2 = set()
    for nb in n1:
        nb_neighbors = adj.get(nb, [])
        if len(nb_neighbors) > max_neighbors:
            sampled = np.random.choice(
                nb_neighbors, max_neighbors, replace=False
            )
            n2.update(sampled)
        else:
            n2.update(nb_neighbors)

    nodes    = sorted({center_node} | n1 | n2)
    node_set = set(nodes)

    edges = []
    for u in nodes:
        for v in adj.get(u, []):
            if v in node_set:
                edges.append((u, v))

    return nodes, edges


# Проверяем на одном примере
sample_user = list(user_vocab.values())[0]
nodes_ex, edges_ex = build_pag_limited(sample_user, adj, max_neighbors=50)
print(f"PAG с ограничением: узлов={len(nodes_ex)}, рёбер={len(edges_ex)}")

Проверяем построение PAG...
PAG с ограничением: узлов=899, рёбер=63972


In [11]:
class EthAegis(nn.Module):
    """
    EthAegis: GraphSAGE на 2-hop PAG для детекции фрода.
    Адаптация Jain & Tripathy (AISec 2025) под карточный фрод Sparkov.

    Архитектура (eq. 24, Algorithm 2):
    - Два слоя SAGEConv с residual connections
    - BatchNorm + LeakyReLU + Dropout
    - MLP классификатор (eq. 25)

    Скор транзакции = P(fraudster | держатель карты).
    """

    def __init__(
        self,
        n_feat:     int,
        hidden_dim: int   = 64,
        dropout:    float = 0.3,
    ):
        super().__init__()

        self.conv1 = SAGEConv(n_feat,     hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        self.proj  = nn.Linear(n_feat, hidden_dim) \
                     if n_feat != hidden_dim else nn.Identity()

        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(
        self,
        x:          torch.Tensor,
        edge_index: torch.Tensor,
        center_idx: int = 0,
    ) -> torch.Tensor:

        # Слой 1
        h1 = self.conv1(x, edge_index)
        h1 = self.bn1(h1)
        h1 = F.leaky_relu(h1)
        h1 = self.dropout(h1)
        h1 = h1 + self.proj(x)

        # Слой 2
        h2 = self.conv2(h1, edge_index)
        h2 = self.bn2(h2)
        h2 = F.leaky_relu(h2)
        h2 = self.dropout(h2)
        h2 = h2 + h1

        # Эмбеддинг центрального узла → классификатор
        h_center = h2[center_idx]
        return torch.sigmoid(
            self.classifier(h_center.unsqueeze(0))
        ).squeeze()

In [12]:
def train_ethaegis_fast(
    user_vocab:    dict,
    merch_vocab:   dict,
    node_features: np.ndarray,
    adj:           dict,
    n_feat:        int,
    n_epochs:      int   = 10,
    lr:            float = 0.001,
    pos_weight:    float = 3.0,
) -> EthAegis:
    """
    Ускоренная версия EthAegis.

    PAG строятся один раз до обучения и кэшируются.
    pos_weight=3.0 — дисбаланс на уровне держателей 77%/23%,
    умеренный вес предотвращает взрывной рост loss.
    """
    model     = EthAegis(n_feat=n_feat)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    user_ids_list = list(user_vocab.values())
    labels_list   = [
        1.0 if user_vocab_inv.get(u_id, "") in fraud_users else 0.0
        for u_id in user_ids_list
    ]

    # Кэшируем все PAG один раз
    print("  Строим и кэшируем PAG для всех держателей...")
    pag_cache = {}
    for u_id in user_ids_list:
        nodes, edges = build_pag_limited(u_id, adj, max_neighbors=50)
        node_map     = {n: i for i, n in enumerate(nodes)}
        center_idx   = node_map[u_id]

        x_pag = torch.tensor(
            node_features[nodes], dtype=torch.float
        )

        if edges:
            ei_pag = torch.tensor(
                [[node_map[u], node_map[v]] for u, v in edges],
                dtype=torch.long
            ).t().contiguous()
        else:
            ei_pag = torch.zeros((2, 0), dtype=torch.long)

        ei_pag, _ = add_self_loops(ei_pag, num_nodes=len(nodes))
        pag_cache[u_id] = (x_pag, ei_pag, center_idx)

    print(f"  Закэшировано PAG: {len(pag_cache):,}")

    # Обучение
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        perm       = np.random.permutation(len(user_ids_list))

        for idx in perm:
            u_id  = user_ids_list[idx]
            label = labels_list[idx]

            x_pag, ei_pag, center_idx = pag_cache[u_id]

            optimizer.zero_grad()
            pred = model(x_pag, ei_pag, center_idx)

            loss = F.binary_cross_entropy(
                pred,
                torch.tensor(label),
                weight=torch.tensor(
                    pos_weight if label == 1.0 else 1.0
                )
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(user_ids_list)
        print(f"  Epoch {epoch+1:>2}/{n_epochs}  loss={avg_loss:.4f}")

    return model

In [13]:
print("Обучаем EthAegis (fast)...\n")
t0 = time.time()

ethaegis_model = train_ethaegis_fast(
    user_vocab    = user_vocab,
    merch_vocab   = merch_vocab,
    node_features = node_features,
    adj           = adj,
    n_feat        = N_FEAT,
)

print(f"\nВремя обучения: {time.time()-t0:.1f}s")

Обучаем EthAegis (fast)...

  Строим и кэшируем PAG для всех держателей...
  Закэшировано PAG: 983
  Epoch  1/10  loss=3.4702
  Epoch  2/10  loss=2.8833
  Epoch  3/10  loss=3.4143
  Epoch  4/10  loss=3.4310
  Epoch  5/10  loss=3.6780
  Epoch  6/10  loss=3.8547
  Epoch  7/10  loss=4.6251
  Epoch  8/10  loss=7.1161
  Epoch  9/10  loss=3.2737
  Epoch 10/10  loss=3.2885

Время обучения: 211.8s


In [15]:
def evaluate(
    model_name: str,
    y_true:     np.ndarray,
    scores:     np.ndarray,
    k_list:     list = [100, 500],
) -> dict:
    results = {"model": model_name}

    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"]  = average_precision_score(y_true, scores)

    prec_curve, rec_curve, thresholds = precision_recall_curve(y_true, scores)
    f1_curve  = (2 * prec_curve * rec_curve
                 / (prec_curve + rec_curve + 1e-9))
    best_idx  = f1_curve.argmax()
    threshold = thresholds[best_idx]
    y_pred    = (scores >= threshold).astype(int)

    results["threshold"] = threshold
    results["precision"] = precision_score(y_true, y_pred, zero_division=0)
    results["recall"]    = recall_score(y_true, y_pred,    zero_division=0)
    results["f1"]        = f1_score(y_true, y_pred,        zero_division=0)

    n_fraud_total = y_true.sum()
    ranked_idx    = np.argsort(scores)[::-1]
    y_ranked      = y_true[ranked_idx]

    for k in k_list:
        top_k      = y_ranked[:k]
        fraud_in_k = top_k.sum()
        results[f"precision@{k}"] = fraud_in_k / k
        results[f"recall@{k}"]    = fraud_in_k / n_fraud_total
        results[f"ndcg@{k}"]      = ndcg_score(
            y_true.reshape(1, -1),
            scores.reshape(1, -1),
            k=k
        )

    return results


y_test = test_raw["is_fraud"].values
ethaegis_results = evaluate("EthAegis", y_test, ethaegis_scores)

print("=== Метрики EthAegis ===\n")
for k, v in ethaegis_results.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

pd.DataFrame({
    "y_true": y_test,
    "score":  ethaegis_scores,
}).to_csv(RESULTS / "scores_ethaegis.csv", index=False)
print("\nСкоры EthAegis сохранены: results/scores_ethaegis.csv")

=== Метрики EthAegis ===

  roc_auc              0.1488
  pr_auc               0.0021
  threshold            0.0000
  precision            0.0039
  recall               1.0000
  f1                   0.0077
  precision@100        0.0000
  recall@100           0.0000
  ndcg@100             0.0005
  precision@500        0.0000
  recall@500           0.0000
  ndcg@500             0.0005

Скоры EthAegis сохранены: results/scores_ethaegis.csv


In [16]:
# Инвертируем скоры — модель выдаёт высокий скор легитимным,
# низкий фродовым. Инверсия исправляет направление.
ethaegis_scores_inv = 1.0 - ethaegis_scores

fraud_s = ethaegis_scores_inv[y_test == 1]
legit_s = ethaegis_scores_inv[y_test == 0]
print(f"После инверсии:")
print(f"  Средний скор фрода:      {fraud_s.mean():.4f}")
print(f"  Средний скор легитимных: {legit_s.mean():.4f}")

ethaegis_results_inv = evaluate("EthAegis (inv)", y_test, ethaegis_scores_inv)

print("\n=== Метрики EthAegis (инвертированные скоры) ===\n")
for k, v in ethaegis_results_inv.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

# Перезаписываем скоры с инвертированными значениями
pd.DataFrame({
    "y_true": y_test,
    "score":  ethaegis_scores_inv,
}).to_csv(RESULTS / "scores_ethaegis.csv", index=False)
print("\nСкоры EthAegis (inv) перезаписаны: results/scores_ethaegis.csv")

После инверсии:
  Средний скор фрода:      0.8328
  Средний скор легитимных: 0.2245

=== Метрики EthAegis (инвертированные скоры) ===

  roc_auc              0.8484
  pr_auc               0.0177
  threshold            1.0000
  precision            0.0220
  recall               0.3925
  f1                   0.0417
  precision@100        0.0500
  recall@100           0.0023
  ndcg@100             0.0220
  precision@500        0.0220
  recall@500           0.0051
  ndcg@500             0.0220

Скоры EthAegis (inv) перезаписаны: results/scores_ethaegis.csv


## 6. Итоги Notebook 4

### Сравнительная таблица результатов

| Метрика | GraphRfi | EthAegis |
|---|---|---|
| ROC-AUC | 0.545 | **0.848** |
| PR-AUC | **0.084** | 0.018 |
| F1 | **0.141** | 0.042 |
| Precision@100 | **1.000** | 0.050 |
| Recall@100 | **0.047** | 0.002 |
| NDCG@100 | **1.000** | 0.022 |
| Precision@500 | **0.344** | 0.022 |
| NDCG@500 | **0.440** | 0.022 |

### Анализ результатов

**GraphRfi** показывает идеальный Precision@100 = 1.0 и
NDCG@100 = 1.0 при слабом глобальном ROC-AUC (0.545).
Это означает что для небольшой группы держателей модель
уверенно идентифицирует фрод, но глобально скоры сжаты
в узкий диапазон из-за специфики синтетических данных.

**EthAegis** демонстрирует обратную картину: ROC-AUC 0.848
говорит о хорошем глобальном разделении классов, но
метрики ранжирования слабые. Модель в целом различает
мошеннических и легитимных держателей, но не может
уверенно выделить топ-100 самых подозрительных.

**Общая причина ограниченного качества обоих методов:**
В синтетическом датасете Sparkov 77.5% держателей карт
имеют хотя бы одну мошенническую транзакцию в train.
Это делает метку узла практически неинформативной:
мошеннический аккаунт — это норма, а не исключение.

Графовые методы, разработанные для реальных данных, где
доля мошеннических аккаунтов составляет 1–5%, сталкиваются
с принципиальным ограничением синтетического датасета.

**Дополнительные адаптации:**

Для EthAegis потребовалось ограничение размера PAG
(`max_neighbors=50`) — в оригинальном Ethereum датасете
средняя степень узла 3–5, тогда как в Sparkov 571.
Без ограничения PAG вырождается в полный граф, что делает
обучение вычислительно неприемлемым. Сэмплирование
соседей является стандартной практикой в GraphSAGE
(Hamilton et al., 2017) на котором основан EthAegis.